# Build the watershed model

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pathlib as pl
import flopy
from flopy.discretization import StructuredGrid

from flopy.utils.gridintersect import GridIntersect
from shapely.geometry import LineString

In [ ]:
import sys
sys.path.insert(0, "../data")  # add Folder_2 path to search list
from defaults import *

In [ ]:
base_dir = "./parallel/base"
split_dir = "./parallel/split"
if not os.path.isdir(base_dir):
   os.makedirs(base_dir)
if not os.path.isdir(split_dir):
   os.makedirs(split_dir)

### Load the topology

In [ ]:
fine_topo = flopy.utils.Raster.load("../data/fine_topo.tif")
fig = plt.figure(figsize=figsize)
_ = fine_topo.plot()

### Structured grid parameters

Set the cell dimensions. This will determine the number of cells in the grid. Setting dx = dy = 2500.0 will lead to 9595 active cells

In [ ]:
dx = dy = 2500.0
nrow = int(Ly / dy) + 1
ncol = int(Lx / dx) + 1

### Read in boundary data

Load the boundary data from `defaults.py` and plot

In [ ]:
boundary_polygon = string2geom(geometry["boundary"])
bp = np.array(boundary_polygon)

sfr_segments = string2geom(geometry["streamseg1"])
sfr_segments = sfr_segments[::-1]

riv_geoms = (
    geometry["streamseg2"],
    geometry["streamseg3"],
    geometry["streamseg4"],
)
riv_segments = [string2geom(rg) for rg in riv_geoms]

In [ ]:
fig = plt.figure(figsize=figsize)
ax = fig.add_subplot()
ax.set_aspect("equal")

riv_colors = ("cyan", "green", "orange", "red")

ax.plot(bp[:, 0], bp[:, 1], "ro-")
sa = np.array(sfr_segments)
ax.plot(sa[:, 0], sa[:, 1], color="black", lw=0.75, marker="o")

for idx, sg in enumerate(riv_segments):
    sa = np.array(sg)
    ax.plot(sa[:, 0], sa[:, 1], color=riv_colors[idx], lw=0.75, marker="o")

### Create a structured grid

In [ ]:
working_grid = StructuredGrid(
    nlay=1,
    delr=np.full(ncol, dx),
    delc=np.full(nrow, dy),
    xoff=0.0,
    yoff=0.0,
    top=np.full((nrow, ncol), 1000.0),
    botm=np.full((1, nrow, ncol), -100.0),
)

set_structured_idomain(working_grid, boundary_polygon)
print("grid data: ", Lx, Ly, nrow, ncol)

### Sample the raw topographic data

In [ ]:
top_wg = fine_topo.resample_to_grid(
    working_grid,
    band=fine_topo.bands[0],
    method="linear",
    extrapolate_edges=True,
)

### Intersect river segments with grid

In [ ]:
ixs_riv, cellids_riv, lengths_riv = intersect_segments(working_grid, riv_segments)

Intersect SFR segments with grid

In [ ]:
ixs = GridIntersect(
        working_grid,
    )
v = ixs.intersect(LineString(sfr_segments), sort_by_cellid=False)
cellids_sfr = v['cellids']

In [ ]:
intersection_rg = np.zeros(working_grid.shape[1:])
for loc in cellids_sfr:
    intersection_rg[loc] = 1
for loc in cellids_riv:
    intersection_rg[loc] = 1


In [ ]:
fig = plt.figure(figsize=figsize)
ax = fig.add_subplot()
pmv = flopy.plot.PlotMapView(modelgrid=working_grid)
ax.set_aspect("equal")
pmv.plot_array(top_wg)
pmv.plot_array(
    intersection_rg,
    masked_values=[
        0,
    ],
    alpha=0.2,
    cmap="Reds_r",
)
pmv.plot_inactive(color_noflow="white")
ax.plot(bp[:, 0], bp[:, 1], "r-")
sa = np.array(sfr_segments)
ax.plot(sa[:, 0], sa[:, 1], "b-")

### Set the idomain value to 2 where the river intersects the grid

In [ ]:
river_locations = working_grid.idomain[0].copy()
index_sfr = tuple(np.array(list(zip(*cellids_sfr))))
river_locations[index_sfr] = 2
index_riv = tuple(np.array(list(zip(*cellids_riv))))
river_locations[index_riv] = 2
working_grid.idomain = river_locations.reshape(1, nrow, ncol)

plt.imshow(working_grid.idomain[0])

### Define the number of layers and the thickness of layer 1

In [ ]:
nlay = 5
dv0 = 5.0
leakance = 1.0 / (0.5 * dv0)  # Kv / b  [1/m]

### Create the SFR data for the reaches

In [ ]:
# Build SFR inputs from the stream/grid intersection results in `v`
# Assumes: v, sg, top_wg, leakance already exist
stream_line = LineString(sfr_segments)

# Collect and sort intersections along stream direction
_reaches = []
for rec in v:
    i, j = rec["cellids"]
    rlen = float(rec["lengths"])
    midpt = rec["ixshapes"].interpolate(0.5, normalized=True)
    dist = float(stream_line.project(midpt))
    cell_top = float(top_wg[i, j])
    _reaches.append((dist, int(i), int(j), rlen, cell_top))
    
_reaches.sort(key=lambda x: x[0])
_reaches[:2]

In [ ]:
# regularize stream bed: water shouldn't run uphill
for r in range(1,len(_reaches)-1):
    bl_prev = _reaches[r-1][4]
    bl = _reaches[r][4]
    bl_next = _reaches[r+1][4]
    if bl > bl_prev:
        # put it in between, or at least flat
        new_bl = min(0.5*(bl_next + bl_prev), bl_prev)
        _reaches[r] = _reaches[r][:4] + (new_bl,)

In [ ]:
nreaches = len(_reaches)

# Hydraulic parameters (edit if needed)
rwid = 25.0   # reach width
rbth = 1.0    # streambed thickness
rhk = 0.01    # streambed hydraulic conductivity
man = 0.015   # Manning's n

# Build MF6 SFR packagedata and connectiondata
sfr_packagedata = []
sfr_connectiondata = []

for rno, (dist, i, j, rlen, cell_top) in enumerate(_reaches):
    # streambed top
    rtp = cell_top - 1.0

    # slope from next reach (or previous for last reach)
    if rno < nreaches - 1:
        dist2, _, _, _, top2 = _reaches[rno + 1]
        dd = max(dist2 - dist, 1.0)
        dz = max(cell_top - top2, 1e-3)
    elif nreaches > 1:
        dist1, _, _, _, top1 = _reaches[rno - 1]
        dd = max(dist - dist1, 1.0)
        dz = max(top1 - cell_top, 1e-3)
    else:
        dd, dz = 1.0, 1e-3

    rgrd = max(dz / dd, 1e-5)
    ncon = int(rno > 0) + int(rno < nreaches - 1)

    # ifno, cellid, rlen, rwid, rgrd, rtp, rbth, rhk, man, ncon, ustrf, ndv
    sfr_packagedata.append(
        (rno, (0, i, j), rlen, rwid, rgrd, rtp, rbth, rhk, man, ncon, 1.0, 0)
    )

    # Connection convention: upstream positive, downstream negative
    conn = [rno]
    if rno > 0:
        conn.append(rno - 1)
    if rno < nreaches - 1:
        conn.append(-(rno + 1))
    sfr_connectiondata.append(conn)

# Stress period data (set inflow at first reach; adjust as needed)
stage_out = 2.0
sfr_perioddata = {0: [(0, "inflow", 1000.0)]}

# Initial stage data (set to streambed top for each reach)
sfr_initialstages = [(rno, float(rec[5])) for rno, rec in enumerate(sfr_packagedata)]

print(f"SFR reaches: {nreaches}")
print("Example packagedata row:", sfr_packagedata[0] if nreaches else None)
print("Example connection row:", sfr_connectiondata[0] if nreaches else None)
print(f"Streambed elevation of last reach: {sfr_packagedata[-1][5]}")

### Create the river drain data

In [ ]:
riv_drn_data = build_drain_data(
    working_grid,
    cellids_riv,
    lengths_riv,
    leakance,
    top_wg,
)

### Create the groundwater discharge drain data

In [ ]:
gw_discharge_data = build_groundwater_discharge_data(
    working_grid,
    leakance,
    top_wg,
)

### Create the top and bottom arrays.

Top array is not used by the model.

In [ ]:
topc = np.zeros((nlay, nrow, ncol), dtype=float)
botm = np.zeros((nlay, nrow, ncol), dtype=float)
dv = dv0
topc[0] = top_wg.copy()
botm[0] = topc[0] - dv
for idx in range(1, nlay):
    dv *= 1.5
    topc[idx] = botm[idx - 1]
    botm[idx] = topc[idx] - dv

#### Print the cell thicknesses

In [ ]:
for k in range(nlay):
    print((topc[k] - botm[k]).mean())

### Create CHD data

In [ ]:
# Index set as (lay, row, col) tuples
chd_cells = [(0, int(i), ncol - 1) for i in range(nrow) if working_grid.idomain[0, i,ncol-1] > 0 and botm[0, i, ncol-1] < stage_out]
spd_chd = {0: [[(c), stage_out] for c in chd_cells]}

### Create idomain and starting head data

In [ ]:
idomain = np.array([working_grid.idomain[0, :, :].copy() for k in range(nlay)])
strt = np.array([top_wg.copy() for k in range(nlay)], dtype=float)

In [ ]:
recharge_data = np.zeros_like(top_wg)
recharge_data[:] = 0.000001

# recharge concentration (same shape as recharge/idomain)
rch_conc = np.full((nrow, ncol), 1.0, dtype=float)

## Build the model files using FloPy
Note that the CSV solver output is enabled. We will use that in one of the other notebooks.

In [ ]:
sim = flopy.mf6.MFSimulation(
    sim_ws=base_dir,
    sim_name="sim",
    exe_name="mf6",
    memory_print_option="summary",
)

tdis = flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(10.0,1,1.0)])

gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname="gwf",
    print_input=False,
    save_flows=True,
    newtonoptions="NEWTON UNDER_RELAXATION",
)

imsgwf = flopy.mf6.ModflowIms(
    sim,
    inner_maximum=200,
    outer_maximum=500,
    print_option="ALL",
    outer_dvclose=1e-5,
    inner_dvclose=1e-6,
    linear_acceleration="BICGSTAB",
    filename=f"{gwf.name}.ims",
)
sim.register_ims_package(imsgwf, [gwf.name])

dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=dx,
    delc=dy,
    idomain=idomain,
    top=top_wg,
    botm=botm,
    xorigin=0.0,
    yorigin=0.0,
)

ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)

chd = flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data=spd_chd,
    pname="CHD-1",
)

npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    icelltype=1,
    k=1.0,
)
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    save_flows=True,
    iconvert=1,
    ss=1e-6,
    sy=0.2,
    steady_state={0: True},
    transient={0: False},
)
rch = flopy.mf6.ModflowGwfrcha(
    gwf,
    pname="RCH-1",
    auxiliary=["concentration"],
    recharge=recharge_data,
    aux=[rch_conc],
)

sfr = flopy.mf6.ModflowGwfsfr(
    gwf,
    filename="sfr.sfr",
    save_flows=True,
    budget_filerecord="sfr.budget.bin",
    print_input=False,    
    stage_filerecord="stages.sfr.bin",
    nreaches=nreaches,
    packagedata=sfr_packagedata,
    connectiondata=sfr_connectiondata,
    initialstages=sfr_initialstages,
    perioddata=sfr_perioddata,
    pname="SFR-1",
)

drn = flopy.mf6.ModflowGwfdrn(
    gwf,
    stress_period_data=riv_drn_data,
    pname="river",
    filename="drn_riv.drn",
)

drn_gwd = flopy.mf6.ModflowGwfdrn(
    gwf,
    auxiliary=["depth"],
    auxdepthname="depth",
    stress_period_data=gw_discharge_data,
    pname="gwd",
    filename="drn_gwd.drn",
)

oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{gwf.name}.hds",
    budget_filerecord=f"{gwf.name}.cbc",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("BUDGET", "ALL")],
)

gwt = flopy.mf6.ModflowGwt(sim, modelname="gwt")

imsgwt = flopy.mf6.ModflowIms(
    sim,
    inner_maximum=100,
    outer_maximum=50,
    print_option="ALL",
    outer_dvclose=1e-5,
    inner_dvclose=1e-6,
    linear_acceleration="BICGSTAB",
    filename=f"{gwt.name}.ims",
)
sim.register_ims_package(imsgwt, [gwt.name])

dis_gwt = flopy.mf6.ModflowGwtdis(
    gwt,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=dx,
    delc=dy,
    idomain=idomain,
    top=top_wg,
    botm=botm,
    xorigin=0.0,
    yorigin=0.0,
)

porosity = 0.2 * np.ones_like(idomain)
mst = flopy.mf6.ModflowGwtmst(
    gwt,
    save_flows=True,
    porosity=porosity,
    pname="mst",
    filename=f"{gwt.name}.mst",
)

ic_gwt = flopy.mf6.ModflowGwtic(gwt, strt=0.0)

adv = flopy.mf6.ModflowGwtadv(gwt, scheme="UTVD")

dsp = flopy.mf6.ModflowGwtdsp(gwt, alh=0.1, alv=0.01, atv=0.01, ath1=0.01, ath2=0.01)

# Instantiate Streamflow Mass Transport package
strm_conc = 0.0
sft_packagedata = []
for irno in range(nreaches):
    t = (irno, strm_conc)
    sft_packagedata.append(t)

sft_perioddata = []
for irno in range(nreaches):
    if irno == 0:
        sft_perioddata.append((irno, "INFLOW", strm_conc))
        
sft = flopy.mf6.ModflowGwtsft(
    gwt,
    pname="sft",
    filename="sft.sft",
    flow_package_name="SFR-1",
    print_input=False,
    concentration_filerecord="sft.ucn",
    budget_filerecord="sft.budget.bin",
    packagedata=sft_packagedata,
    reachperioddata=sft_perioddata,
)

ssm = flopy.mf6.ModflowGwtssm(gwt, sources=[["rch-1", "aux", "concentration"]])

oc_gwt = flopy.mf6.ModflowGwtoc(
    gwt,
    concentration_filerecord=f"{gwt.name}.ucn",
    budget_filerecord=f"{gwt.name}.cbc",
    saverecord=[("CONCENTRATION", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("BUDGET", "ALL")],
)

flopy.mf6.ModflowGwfgwt(
    sim,
    exgtype="GWF6-GWT6",
    exgmnamea=gwf.name,
    exgmnameb=gwt.name,
    filename=f"{sim.name}.gwfgwt",
    )


### Count the number of active cells

In [ ]:
ncells, nactive = get_simulation_cell_count(sim)
print("nr. of cells:", ncells, ", active:", nactive)

### Write the model files

In [ ]:
sim.write_simulation()

### Run the model

In [ ]:
sim.run_simulation()

### Plot results

In [ ]:
# Contour map of simulated head (layer 1)
t_plot = -1
gwf_model = sim.get_model("gwf")
hds_file = pl.Path(base_dir) / f"{gwf_model.name}.hds"
hds_obj = flopy.utils.HeadFile(hds_file)
totim = hds_obj.get_times()[t_plot]
head = hds_obj.get_data(totim=totim)

xcc = working_grid.xcellcenters
ycc = working_grid.ycellcenters

h0 = np.ma.masked_where(working_grid.idomain[0] < 1, head[0])

fig, ax = plt.subplots(figsize=figsize)
cf = ax.contourf(xcc, ycc, h0, levels=20, cmap="viridis")
cs = ax.contour(xcc, ycc, h0, levels=20, colors="k", linewidths=0.4)
ax.clabel(cs, inline=True, fontsize=7, fmt="%.1f")

ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Head contour map (layer 1, t={totim})")
plt.colorbar(cf, ax=ax, label="Head", shrink=0.35)
plt.tight_layout()
plt.show()

In [ ]:
# Plot distance to groundwater (depth to water table) as a plan view
fig, ax = plt.subplots(figsize=figsize)
ax.set_aspect("equal")

# Calculate depth to water table (top elevation - head)
depth_to_water = top_wg - head[0]

pmv = flopy.plot.PlotMapView(modelgrid=working_grid, ax=ax, layer=0)
pa = pmv.plot_array(depth_to_water, cmap="RdYlBu_r")
pmv.plot_inactive(color_noflow="white")

cbar = plt.colorbar(pa, ax=ax, label="Depth to Water Table (m)", shrink=0.35)
ax.plot(bp[:, 0], bp[:, 1], "k-", linewidth=1.5, label="Domain boundary")
ax.set_title(f"Distance to Groundwater (t={totim})")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# nearest model row for requested y
t_plot = -1
y_target = 51000
row_idx = int(np.argmin(np.abs(ycc[:, 0] - y_target)))
y_sel = float(ycc[row_idx, 0])

gwf_model = sim.get_model("gwf")

# read head at last time
hds_file = pl.Path(base_dir) / f"{gwf_model.name}.hds"
hds_obj = flopy.utils.HeadFile(hds_file)
totim = hds_obj.get_times()[t_plot]
head = hds_obj.get_data(totim=totim)

# read specific discharge from budget output at same time
cbc_file = pl.Path(base_dir) / f"{gwf_model.name}.cbc"
cbc_obj = flopy.utils.CellBudgetFile(cbc_file)
spdis = cbc_obj.get_data(text="DATA-SPDIS", totim=totim)[0]
qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf_model)

# plot cross section
fig, ax = plt.subplots(figsize=(8,4))
xsect = flopy.plot.PlotCrossSection(model=gwf_model, ax=ax, line={"row": row_idx})

pc = xsect.plot_array(head, cmap="viridis")
xsect.plot_grid(lw=0.2, color="0.6")
xsect.plot_inactive(color_noflow="white")
xsect.plot_vector(qx, qy, qz, normalize=True, color="k", hstep=2, kstep=1)

plt.colorbar(pc, ax=ax, label="Head")
ax.set_title(f"Cross section at y={y_sel:.1f} (requested {y_target:.1f}), time={totim}")
ax.set_xlabel("x")
ax.set_ylabel("elevation")
plt.tight_layout()
plt.show()

In [ ]:
chainage = np.cumsum([val[2] for val in sfr_packagedata])
stage_file = pl.Path(base_dir) / "stages.sfr.bin"
stage_obj = flopy.utils.HeadFile(stage_file, text="stage")
times = stage_obj.get_times()
stage = np.asarray(stage_obj.get_data(totim=times[t_plot])).squeeze()

n = min(len(chainage), stage.size)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(chainage[:n], stage[:n], "o-", lw=1.2, ms=3)
ax.set_xlabel("Chainage (m)")
ax.set_ylabel("River stage (m)")
ax.set_title(f"SFR stage vs chainage (t={times[t_plot]})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from flopy.mf6.utils.model_splitter import Mf6Splitter

In [ ]:
npartitions = 3
splitter = Mf6Splitter(sim)
mask = splitter.optimize_splitting_mask(nparts=npartitions)
split_sim = splitter.split_multi_model(mask)
split_sim.set_sim_path(split_dir)

In [ ]:
# to enable the transport mover in the exchange
exgs = split_sim.get_package("gwtgwt")
for idx, exg in enumerate(exgs): 
    exg.mvt_filerecord = f"exgmvt_{idx}.mvt"
    exg.mvt.initialize(filename=exg.mvt_filerecord)

In [ ]:
split_sim.write_simulation()
split_sim.run_simulation(processors=npartitions)

In [ ]:
#Read SFT concentration output
t_plot = -1
gwt = sim.get_model("gwt")
sft_conc_file = pl.Path(base_dir) / "sft.ucn"

fig = plt.figure(figsize=figsize)
sft_conc_obj = flopy.utils.HeadFile(sft_conc_file, text="concentration")
times = sft_conc_obj.get_times()
sft_conc = sft_conc_obj.get_data(totim=times[t_plot])
plt.plot(chainage, sft_conc.squeeze(), "o-", label="serial")

idx = 0
idx_seg = 0
for mname in split_sim.model_names:
    if "gwt" not in mname:
        continue
    sft_conc_file = pl.Path(split_dir) / f"sft_{idx}.ucn"
    sft_conc_obj = flopy.utils.HeadFile(sft_conc_file, text="concentration")
    sft_conc = sft_conc_obj.get_data(totim=times[t_plot])
    sft_conc_part = sft_conc.squeeze().tolist()
    nseg = len(sft_conc_part)
    plt.plot(chainage[idx_seg : idx_seg+nseg], sft_conc_part, label=f"domain {idx + 1}")
    idx_seg = idx_seg + nseg
    idx = idx + 1

plt.suptitle("concentration vs. chainage")
plt.legend()


In [ ]:
# from matplotlib.collections import LineCollection

# # Plot 2D GWT concentration with SFR stream concentration overlay
# t_plot = 0
# gwt_model = sim.get_model("gwt")
# conc_file = pl.Path(base_dir) / "gwt.ucn"
# conc_obj = flopy.utils.HeadFile(conc_file, text="concentration")
# times = conc_obj.get_times()
# conc = conc_obj.get_data(totim=times[t_plot])

# fig, ax = plt.subplots(figsize=figsize)
# ax.set_aspect("equal")
# pmv = flopy.plot.PlotMapView(modelgrid=working_grid, ax=ax, layer=0)
# pa = pmv.plot_array(conc[0], cmap="viridis")
# pmv.plot_inactive(color_noflow="white")
# plt.colorbar(pa, ax=ax, label="GWT Concentration", shrink=0.35)

# # Overlay SFR stream concentration
# sft_conc_file_base = pl.Path(base_dir) / "sft.ucn"
# sft_conc_obj_base = flopy.utils.HeadFile(sft_conc_file_base, text="concentration")
# sft_conc_base = sft_conc_obj_base.get_data(totim=times[t_plot]).squeeze()

# # Get cell centers for each SFR reach
# xcc = working_grid.xcellcenters
# ycc = working_grid.ycellcenters
# sfr_x = [xcc[int(rec[1][1]), int(rec[1][2])] for rec in sfr_packagedata]
# sfr_y = [ycc[int(rec[1][1]), int(rec[1][2])] for rec in sfr_packagedata]

# # Cross section at a chosen y-coordinate
# y_target = 50000.0  # <-- change this to any y value of interest
# ax.axhline(y=y_target, color="red", linestyle="--", linewidth=1.0, label=f"y={y_target:.0f}")

# sc = ax.scatter(sfr_x, sfr_y, c=sft_conc_base, cmap="plasma",
#                 s=40, zorder=5, edgecolors="k", linewidths=1.0)
# plt.colorbar(sc, ax=ax, label="SFT Stream Concentration", shrink=0.35)

# ax.set_title(f"Stream concentration overlay (t={times[t_plot]})")
# plt.tight_layout()
# plt.show()